# Day 002 — Singly Linked List · Selection Sort · Jump Search
**Date:** 2026-08-03  |  **Difficulty:** Beginner  |  **Series:** Daily DSA

---

## What You Will Learn Today
| Topic | Concept | Time Complexity |
|-------|---------|----------------|
| Data Structure | **Singly Linked List** | Access O(n), Insert/Delete at head O(1) |
| Sorting | **Selection Sort** | Always O(n²) — at most n−1 swaps |
| Searching | **Jump Search** | O(√n) — beats Linear on sorted arrays |

> **Recap from Day 001:** Array gives O(1) random access but O(n) insert/delete.  
> Today's Linked List *flips* that trade-off: O(1) head insert/delete, O(n) access.

> Run each cell top-to-bottom with **Shift+Enter** to follow along interactively.

---
## PART 1 — Data Structure: Singly Linked List

A **linked list** is a chain of **nodes**, each containing a value and a pointer (`next`) to the following node.  
Unlike an array, nodes live anywhere in memory — there is **no contiguous block**.

### Memory layout
```
head
 │
 ▼
┌──────┬──────┐   ┌──────┬──────┐   ┌──────┬──────┐   ┌──────┬──────┐
│  10  │  ●───┼──▶│  20  │  ●───┼──▶│  30  │  ●───┼──▶│  40  │ None │
└──────┴──────┘   └──────┴──────┘   └──────┴──────┘   └──────┴──────┘
 Node 0            Node 1            Node 2            Node 3 (tail)
```

### Trade-offs vs Array
| Operation | Array | Linked List |
|-----------|-------|-------------|
| Access by index | **O(1)** | O(n) |
| Insert / delete at **head** | O(n) | **O(1)** |
| Insert / delete at **tail** | O(1)* | O(n)** |
| Insert / delete in **middle** | O(n) | O(n) |
| Extra memory per element | none | one pointer per node |

\* amortised for dynamic arrays  
\** O(1) if you keep a tail pointer

In [ ]:
# ─── Singly Linked List ───────────────────────────────────────────────────────

class Node:
    """A single node holding data and a reference to the next node."""
    __slots__ = ('data', 'next')   # saves memory vs a regular __dict__

    def __init__(self, data):
        self.data = data
        self.next = None

    def __repr__(self):
        return f'Node({self.data})'


class SinglyLinkedList:
    """
    Singly Linked List with head and tail pointers.
    Keeping a tail pointer makes append() O(1) instead of O(n).
    """

    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    # ── O(1) ──────────────────────────────────────────────────────────────────
    def prepend(self, data):
        """Insert a new node at the head — O(1)."""
        node = Node(data)
        node.next = self.head
        self.head = node
        if self.tail is None:      # first element
            self.tail = node
        self._size += 1

    def append(self, data):
        """Insert a new node at the tail — O(1) thanks to tail pointer."""
        node = Node(data)
        if self.tail is None:      # empty list
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self._size += 1

    def pop_head(self):
        """Remove and return the head value — O(1)."""
        if self.head is None:
            raise IndexError('pop from empty list')
        value = self.head.data
        self.head = self.head.next
        if self.head is None:
            self.tail = None
        self._size -= 1
        return value

    # ── O(n) ──────────────────────────────────────────────────────────────────
    def insert_after(self, target_data, new_data):
        """Insert new_data right after the first node whose data == target_data."""
        current = self.head
        while current:
            if current.data == target_data:
                node = Node(new_data)
                node.next = current.next
                current.next = node
                if current is self.tail:   # inserted after tail → new tail
                    self.tail = node
                self._size += 1
                return
            current = current.next
        raise ValueError(f'{target_data!r} not found in list')

    def delete(self, target_data):
        """Remove the first node whose data == target_data — O(n)."""
        prev, current = None, self.head
        while current:
            if current.data == target_data:
                if prev:                   # not the head
                    prev.next = current.next
                else:                      # removing head
                    self.head = current.next
                if current.next is None:   # removed tail
                    self.tail = prev
                self._size -= 1
                return
            prev, current = current, current.next
        raise ValueError(f'{target_data!r} not found in list')

    def get(self, index):
        """Return value at index — O(n)."""
        if not (0 <= index < self._size):
            raise IndexError(f'Index {index} out of range')
        current = self.head
        for _ in range(index):
            current = current.next
        return current.data

    def reverse(self):
        """Reverse the list in-place — O(n), O(1) space."""
        prev, current = None, self.head
        self.tail = self.head          # old head becomes new tail
        while current:
            nxt = current.next
            current.next = prev
            prev = current
            current = nxt
        self.head = prev

    def to_list(self):
        """Convert to a plain Python list — useful for printing / sorting."""
        result, current = [], self.head
        while current:
            result.append(current.data)
            current = current.next
        return result

    def __len__(self):  return self._size
    def __repr__(self): return ' → '.join(str(x) for x in self.to_list()) + ' → None'

In [ ]:
# ─── Demo ─────────────────────────────────────────────────────────────────────
ll = SinglyLinkedList()

print("--- Building the list ---")
for v in [20, 30, 40]:
    ll.append(v)
ll.prepend(10)
print(f"After append(20,30,40) + prepend(10) : {ll}")
print(f"  head={ll.head.data}  tail={ll.tail.data}  size={len(ll)}")

print("\n--- O(n) mutations ---")
ll.insert_after(20, 25)
print(f"insert_after(20, 25)               : {ll}")
ll.delete(25)
print(f"delete(25)                         : {ll}")
print(f"get(2)                             : {ll.get(2)}")

print("\n--- Reverse ---")
ll.reverse()
print(f"After reverse()                    : {ll}")
ll.reverse()   # restore

print("\n--- pop_head ---")
print(f"pop_head() → {ll.pop_head()}")
print(f"List now   : {ll}")

---
## PART 2 — Sorting Algorithm: Selection Sort

### Core Idea
Divide the array into a **sorted left region** and an **unsorted right region**.  
On each pass, **select the minimum** element from the unsorted region and **swap** it into the next sorted position.  
After `i` passes, the first `i` elements are permanently in their correct positions.

### Step-by-step on `[64, 25, 12, 22, 11]`
```
Pass 1: min of [64,25,12,22,11]=11 → swap(64,11) → [11 | 25,12,22,64]
Pass 2: min of [25,12,22,64]=12   → swap(25,12) → [11,12 | 22,25,64]  ← wait, wrong
         Actually min of [25,12,22,64]=12 at idx2 → swap(25,12) → [11,12 | 22,25,64]
Pass 3: min of [22,25,64]=22      → already in place → [11,12,22 | 25,64]
Pass 4: min of [25,64]=25         → already in place → [11,12,22,25 | 64]
Done  → [11, 12, 22, 25, 64] ✓
```

### Key property
Selection Sort performs **at most n−1 swaps** — the minimum possible for a comparison sort.  
This makes it ideal when write operations are expensive (e.g., flash memory).

### Complexity
| Case | Time | Space | Swaps |
|------|------|-------|-------|
| Best | O(n²) | O(1) | O(n) |
| Average | O(n²) | O(1) | O(n) |
| Worst | O(n²) | O(1) | O(n) |

> Unlike Bubble Sort, Selection Sort has **no early-exit** — it always does n(n-1)/2 comparisons regardless of input order.

In [ ]:
# ─── Selection Sort ───────────────────────────────────────────────────────────

def selection_sort(arr, verbose=False):
    """
    Sort `arr` in-place using Selection Sort.

    Algorithm:
      for i in 0..n-2:
          find the index of the minimum element in arr[i..n-1]
          swap arr[i] with arr[min_idx]   (skipped if already in place)

    Args:
        arr     : list of comparable elements
        verbose : print each pass if True

    Returns:
        (sorted list, swap_count, comparison_count)
    """
    a = arr[:]          # copy — leave original unchanged
    n = len(a)
    swaps = comparisons = 0

    for i in range(n - 1):
        min_idx = i

        # Scan unsorted region for the minimum
        for j in range(i + 1, n):
            comparisons += 1
            if a[j] < a[min_idx]:
                min_idx = j

        # Only swap if the minimum isn't already in position
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]
            swaps += 1

        if verbose:
            marker = f"swap({arr[i] if i < len(arr) else '?'}, {a[i]})" if min_idx != i else "no swap"
            print(f"  Pass {i+1}: min_idx={min_idx}  {marker}  → {a}")

    return a, swaps, comparisons

In [ ]:
# ─── Verbose walkthrough ──────────────────────────────────────────────────────
sample = [64, 25, 12, 22, 11]
print(f"Input : {sample}")
sorted_arr, swaps, cmps = selection_sort(sample, verbose=True)
print(f"Output: {sorted_arr}")
print(f"Stats : {swaps} swaps, {cmps} comparisons  (n(n-1)/2 = {len(sample)*(len(sample)-1)//2})")

In [ ]:
# ─── Test Cases ───────────────────────────────────────────────────────────────
test_cases = [
    ([64, 25, 12, 22, 11],   "random"),
    ([1, 2, 3, 4, 5],        "already sorted"),
    ([5, 4, 3, 2, 1],        "reverse sorted"),
    ([42],                   "single element"),
    ([],                     "empty list"),
    ([3, 3, 1, 1, 2],        "with duplicates"),
    ([-7, 0, 4, -2, 9],      "with negatives"),
]

print(f"{'Input':<30} {'Sorted':<30} {'Swaps':>5}  {'Case'}")
print("-" * 80)
for data, label in test_cases:
    result, swaps, _ = selection_sort(data)
    print(f"{str(data):<30} {str(result):<30} {swaps:>5}  {label}")

---
## PART 3 — Searching Algorithm: Jump Search

### Core Idea
Jump Search works on a **sorted array** by jumping ahead in fixed steps of size **√n**.  
Once we overshoot (find an element ≥ target), we do a **linear scan backwards** in the previous block.

```
Array : [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233]  (n=14, step=√14≈3)
Target: 55

Jump 1: check index 0  → 0  < 55  → jump
Jump 2: check index 3  → 2  < 55  → jump
Jump 3: check index 6  → 8  < 55  → jump
Jump 4: check index 9  → 34 < 55  → jump
Jump 5: check index 12 → 144 ≥ 55 → OVERSHOOT — linear scan [9..11]
          index 9=34, index 10=55 ✓  → return 10
```

### Why √n is the optimal step size
- Jumps needed ≈ n/step
- Linear scan in last block ≈ step
- Total = n/step + step, minimised at step = √n → O(√n)

### Complexity
| Case | Time | Space |
|------|------|-------|
| Best (first jump) | O(1) | O(1) |
| Average | O(√n) | O(1) |
| Worst | O(√n) | O(1) |

**vs Linear Search:** O(√n) beats O(n) on large sorted arrays.  
**vs Binary Search:** O(√n) is worse than O(log n) but Jump Search only moves **forward**, making it better for magnetic tapes or sequential media where backward seeks are expensive.

In [ ]:
# ─── Jump Search ──────────────────────────────────────────────────────────────
import math

def jump_search(arr, target, verbose=False):
    """
    Search for `target` in a SORTED array using Jump Search.

    Args:
        arr     : sorted list of comparable elements
        target  : value to find
        verbose : print each jump/scan step if True

    Returns:
        int : index of target, or -1 if not found
    """
    n = len(arr)
    if n == 0:
        return -1

    step = int(math.sqrt(n))   # optimal block size
    prev = 0

    # ── Phase 1: Jump forward until we overshoot or hit the end ───────────────
    while prev < n and arr[min(step, n) - 1] < target:
        if verbose:
            print(f"  Jump: checking index {min(step,n)-1} → {arr[min(step,n)-1]} < {target}")
        prev = step
        step += int(math.sqrt(n))
        if prev >= n:
            if verbose: print("  Jumped past end — not found")
            return -1

    # ── Phase 2: Linear scan backwards in the current block ───────────────────
    block_end = min(step, n)
    if verbose:
        print(f"  Overshot at index {block_end-1}={arr[block_end-1]} ≥ {target}")
        print(f"  Linear scan from index {prev} to {block_end-1}")

    for i in range(prev, block_end):
        if verbose:
            print(f"    scan index {i} → {arr[i]}")
        if arr[i] == target:
            return i
        if arr[i] > target:    # sorted — can stop early
            break

    return -1

In [ ]:
# ─── Verbose walkthrough ──────────────────────────────────────────────────────
fibs = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233]
print(f"Array : {fibs}")
print(f"n={len(fibs)}, step=√{len(fibs)}≈{int(math.sqrt(len(fibs)))}\n")

print("Searching for 55:")
idx = jump_search(fibs, 55, verbose=True)
print(f"Result: index {idx} → value {fibs[idx] if idx != -1 else 'N/A'}")

print("\nSearching for 100 (not present):")
idx = jump_search(fibs, 100, verbose=True)
print(f"Result: {idx}  (not found)")

In [ ]:
# ─── Test Cases ───────────────────────────────────────────────────────────────
# Note: array MUST be sorted for Jump Search to work correctly
search_tests = [
    ([1,3,5,7,9,11,13,15,17,19], 7,   "found in first block"),
    ([1,3,5,7,9,11,13,15,17,19], 1,   "found at start"),
    ([1,3,5,7,9,11,13,15,17,19], 19,  "found at end"),
    ([1,3,5,7,9,11,13,15,17,19], 6,   "not in list (between elements)"),
    ([1,3,5,7,9,11,13,15,17,19], 99,  "not in list (beyond end)"),
    ([42],                         42,  "single element — found"),
    ([42],                         1,   "single element — not found"),
    ([],                           5,   "empty list"),
    (list(range(0, 100, 2)),       64,  "100-element range, target present"),
]

print(f"{'Array (preview)':<38} {'Target':>6}  {'Result':<14} {'Case'}")
print("-" * 85)
for arr, target, label in search_tests:
    result = jump_search(arr, target)
    preview = str(arr[:5]) + ('...' if len(arr) > 5 else '')
    found   = f"index {result}" if result != -1 else "not found"
    print(f"{preview:<38} {str(target):>6}  {found:<14} {label}")

---
## PART 4 — Putting It All Together

Real-world mini-pipeline:
1. Build a **Singly Linked List** of employee IDs
2. Convert to array and sort with **Selection Sort**
3. Search for a specific ID with **Jump Search**

In [ ]:
# ─── End-to-End Example ───────────────────────────────────────────────────────
import random
random.seed(7)

# 1. Linked list of employee IDs (arrived in random order)
roster = SinglyLinkedList()
raw_ids = [random.randint(1000, 9999) for _ in range(12)]
for emp_id in raw_ids:
    roster.append(emp_id)
print("1. Roster (insertion order):")
print("  ", roster)

# 2. Sort with Selection Sort
sorted_ids, swaps, cmps = selection_sort(roster.to_list())
print(f"\n2. After Selection Sort ({swaps} swaps, {cmps} comparisons):")
print("  ", sorted_ids)

# 3. Jump Search for a known ID
target = sorted_ids[7]   # pick one that definitely exists
pos = jump_search(sorted_ids, target)
print(f"\n3. Jump Search for ID {target}:")
print(f"   Found at sorted index {pos}")

# 4. Search for a non-existent ID
missing = 5555
pos2 = jump_search(sorted_ids, missing)
print(f"\n4. Jump Search for ID {missing}: {'not found' if pos2==-1 else pos2}")

# 5. Summary
print("\n--- Summary ---")
print(f"   Total employees : {len(sorted_ids)}")
print(f"   Lowest ID       : {sorted_ids[0]}")
print(f"   Highest ID      : {sorted_ids[-1]}")

---
## Complexity Cheat Sheet

```
┌──────────────────────────┬──────────┬──────────┬──────────┬─────────┐
│ Operation                │ Best     │ Average  │ Worst    │ Space   │
├──────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Linked List — prepend    │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
│ Linked List — append*    │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
│ Linked List — pop_head   │ O(1)     │ O(1)     │ O(1)     │ O(1)   │
│ Linked List — access/del │ O(n)     │ O(n)     │ O(n)     │ O(1)   │
│ Linked List — reverse    │ O(n)     │ O(n)     │ O(n)     │ O(1)   │
├──────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Selection Sort           │ O(n²)    │ O(n²)    │ O(n²)    │ O(1)   │
│   (swaps)                │ O(1)     │ O(n)     │ O(n)     │        │
├──────────────────────────┼──────────┼──────────┼──────────┼─────────┤
│ Jump Search              │ O(1)     │ O(√n)    │ O(√n)    │ O(1)   │
└──────────────────────────┴──────────┴──────────┴──────────┴─────────┘
 * with tail pointer
```

## Key Takeaways
- Linked List shines when you need **frequent head insertions/deletions** with no random access.
- Selection Sort's **minimal swap count** (≤ n−1) is its unique edge over Bubble/Insertion Sort.
- Jump Search hits the sweet spot between Linear O(n) and Binary O(log n) — great for **forward-only media**.

---
**Tomorrow — Day 003:** Doubly Linked List · Insertion Sort · Binary Search